le nom: El jattioui
le prenom: Maryame
Master:GLCC

In [1]:
import numpy as np

# =================================================================
# 1. DATASET D'ENTRAÎNEMENT (Input Data)
# =================================================================
# Nous utilisons des points en 2D. 
# Les groupes sont définis par leur proximité (densité).
X_train = np.array([
    [1, 2], [1.5, 1.8], [2, 2], [8, 7], [8, 8], [8.5, 7.5], 
    [25, 80], [1, 1], [0.5, 1], [50, 50] 
])
# Note : [25, 80] et [50, 50] sont très isolés, ils devraient être du "Bruit".

# =================================================================
# 2. L'ALGORITHME DBSCAN (Density-Based Clustering)
# =================================================================
class MyDBSCAN:
    """
    DBSCAN : Contrairement à K-Means, cet algorithme ne demande pas le 
    nombre de clusters 'k'. Il découvre les clusters par densité.
    """
    def __init__(self, eps=2.0, min_samples=3):
        """
        :param eps: Rayon de voisinage (Epsilon). Si la distance entre deux 
                    points est <= eps, ils sont voisins.
        :param min_samples: Nombre minimum de voisins pour qu'un point 
                            devienne un 'Core Point' (Point central).
        """
        self.eps = eps
        self.min_samples = min_samples
        self.labels = None # Liste des résultats (ID du cluster ou -1 pour bruit)

    def _compute_distance(self, p1, p2):
        """ Calcule la distance euclidienne entre deux points. """
        return np.sqrt(np.sum((p1 - p2) ** 2))

    def _get_neighbors(self, X, target_idx):
        """ 
        Identifie tous les points situés dans le cercle de rayon 'eps' 
        autour du point cible. 
        """
        neighbors = []
        for i in range(len(X)):
            if self._compute_distance(X[target_idx], X[i]) <= self.eps:
                neighbors.append(i)
        return neighbors

    def fit(self, X):
        """
        Logique principale de partitionnement.
        États des labels :
        -2 : Non visité
        -1 : Bruit (Noise)
         0, 1, 2... : ID du cluster
        """
        n_samples = len(X)
        self.labels = np.full(n_samples, -2) # Initialisation à 'Non visité'
        cluster_id = 0

        # On parcourt chaque point du dataset
        for i in range(n_samples):
            # Si le point a déjà été assigné à un cluster ou marqué bruit, on l'ignore
            if self.labels[i] != -2:
                continue

            # ÉTAPE 1 : Trouver les voisins directs
            neighbors = self._get_neighbors(X, i)

            # ÉTAPE 2 : Vérifier si c'est un point central (Core Point)
            if len(neighbors) < self.min_samples:
                # Trop isolé : on le marque comme 'Bruit' (peut changer plus tard)
                self.labels[i] = -1
            else:
                # C'est un Core Point ! On commence à construire un nouveau cluster
                self._expand_cluster(X, i, neighbors, cluster_id)
                cluster_id += 1 # On incrémente pour le prochain groupe trouvé

        return self.labels

    def _expand_cluster(self, X, core_idx, neighbors, cluster_id):
        """
        Algorithme de propagation (similaire à un parcours en largeur/BFS).
        On explore tous les voisins, et les voisins de leurs voisins.
        """
        # On assigne le point central au cluster actuel
        self.labels[core_idx] = cluster_id
        
        # 'queue' contient la liste des points à explorer (graines du cluster)
        queue = list(neighbors)
        
        # On parcourt la file d'attente dynamiquement
        idx = 0
        while idx < len(queue):
            point_idx = queue[idx]

            # Cas A : Le point était marqué comme bruit
            # Il devient un 'Border Point' (Point de bordure) car il touche un Core Point
            if self.labels[point_idx] == -1:
                self.labels[point_idx] = cluster_id
            
            # Cas B : Le point n'a jamais été visité
            elif self.labels[point_idx] == -2:
                # On l'ajoute au cluster
                self.labels[point_idx] = cluster_id
                
                # On cherche ses propres voisins pour voir s'il peut propager le cluster
                new_neighbors = self._get_neighbors(X, point_idx)
                
                # Si ce voisin est aussi un Core Point, on ajoute ses voisins à la file
                if len(new_neighbors) >= self.min_samples:
                    queue.extend(new_neighbors)
            
            idx += 1 # On passe au point suivant dans la file

# =================================================================
# 3. EXÉCUTION ET TESTS
# =================================================================
if __name__ == "__main__":
    # Paramètres : eps=2.0 (distance courte), min_samples=3 (densité moyenne)
    model = MyDBSCAN(eps=2.5, min_samples=3)

    # Lancement de l'algorithme
    results = model.fit(X_train)

    print(f"Rapport DBSCAN (eps={model.eps}, min_pts={model.min_samples})")
    print("-" * 50)
    
    for i in range(len(X_train)):
        label = results[i]
        label_str = f"Cluster {label}" if label != -1 else "Bruit (Anomalie)"
        print(f"Point {X_train[i]} -> {label_str}")

    # Conclusion théorique
    n_clusters = len(set(results)) - (1 if -1 in results else 0)
    print("-" * 50)
    print(f"Nombre de clusters détectés : {n_clusters}")

Rapport DBSCAN (eps=2.5, min_pts=3)
--------------------------------------------------
Point [1. 2.] -> Cluster 0
Point [1.5 1.8] -> Cluster 0
Point [2. 2.] -> Cluster 0
Point [8. 7.] -> Cluster 1
Point [8. 8.] -> Cluster 1
Point [8.5 7.5] -> Cluster 1
Point [25. 80.] -> Bruit (Anomalie)
Point [1. 1.] -> Cluster 0
Point [0.5 1. ] -> Cluster 0
Point [50. 50.] -> Bruit (Anomalie)
--------------------------------------------------
Nombre de clusters détectés : 2
